# Module 01 — What Is PyTorch, and Your First Tensors

**Prerequisites:** Module 00 (Prerequisites & Python Refresher)
**Time:** ~45 minutes

## Learning Objectives

- Explain what PyTorch is and why it exists (in one sentence you could say to a colleague).
- Create tensors in several different ways.
- Inspect a tensor's `shape`, `dtype`, and `device`, and explain what each one means.
- Understand *why* PyTorch uses a special `Tensor` object instead of plain Python lists or NumPy arrays.


## Why PyTorch Exists

Suppose you want to train a neural network. You need three things, all at once:

1. **Fast numerical computation** on large arrays of numbers (millions to billions of them).
2. **Automatic differentiation** — a way to compute gradients (the ingredient gradient descent needs) without you hand-deriving calculus for every model.
3. **GPU support**, because a modern GPU can perform the same arithmetic 50-100x faster than a CPU for the kind of embarrassingly parallel math neural networks require.

NumPy gives you (1). Nothing about NumPy gives you (2) or (3) — a NumPy array has no idea it's part of a larger computation, and it cannot run on a GPU.

PyTorch's core contribution is a single object, `torch.Tensor`, that behaves like a NumPy array but *additionally* knows how to:

- live on a GPU,
- track the sequence of operations applied to it, so gradients can be computed automatically later.

That's it. That one idea — an array that can live on a GPU and remembers its own history — is what everything else in this course is built on top of.

PyTorch was released by Meta (then Facebook) AI Research in 2017. It won out over earlier frameworks in large part because it runs your code *immediately*, line by line ("eager execution"), rather than requiring you to build an abstract computation graph before running anything. That means ordinary Python debugging tools — `print()`, breakpoints, `if`/`else` — work exactly as you'd expect.


## Setting Up

Run the cell below. If PyTorch isn't installed, uncomment the `pip install` line.


In [1]:
# !pip install torch --quiet

import torch
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())


PyTorch version: 2.2.2+cu121
GPU available: True


## What Is a Tensor?

A **tensor** is a container for numbers, arranged along zero or more dimensions. That's the whole definition — the rest is vocabulary for describing *how* the numbers are arranged.

| Name | Dimensions | Example |
|---|---|---|
| Scalar | 0 | a single number, `7.0` |
| Vector | 1 | a list of numbers, `[1, 2, 3]` |
| Matrix | 2 | a grid of numbers (rows × columns) |
| Tensor (general) | 3+ | a stack of matrices, a batch of images, etc. |

In casual PyTorch usage, everyone just says "tensor" regardless of how many dimensions it has — a scalar and an image batch are both "tensors," just with different shapes.

Let's create a few.


In [2]:
# From a plain Python list
x = torch.tensor([1, 2, 3])
print(x)

# A 2D tensor (matrix) from a nested list
y = torch.tensor([[1, 2], [3, 4]])
print(y)

# Tensors filled with zeros / ones of a given shape
z = torch.zeros(2, 3)
w = torch.ones(3)
print(z)
print(w)

# Random values, useful for quickly testing shapes before you have real data
r = torch.randn(2, 3)   # randn = random values from a standard normal distribution
print(r)


tensor([1, 2, 3])
tensor([[1, 2],
        [3, 4]])
tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([1., 1., 1.])
tensor([[ 1.6869, -2.1247, -0.2817],
        [-1.0932,  0.0483, -0.1235]])


### Why not just use a Python list?

You *could* store `[1, 2, 3]` as a plain list. Here's what you'd lose:

- **No vectorized math.** `[1, 2, 3] * 2` on a Python list gives `[1, 2, 3, 1, 2, 3]` (list repetition!), not element-wise multiplication. You'd need a manual loop.
- **No shape/dtype/device metadata.** A list doesn't know it represents a "3x224x224 RGB image" — it's just numbers.
- **No GPU support.** Lists live in regular Python memory and can't be transferred to a GPU.
- **No autograd.** Lists can't record the operations applied to them for later gradient computation.

Tensors solve all four.


## The Three Properties Every Tensor Has

Every tensor has exactly three things worth checking whenever something goes wrong: **shape**, **dtype**, and **device**.


In [3]:
x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

print("shape :", x.shape)    # the size along each dimension
print("dtype :", x.dtype)    # what kind of number is stored (float32, int64, ...)
print("device:", x.device)   # where the data physically lives (cpu or cuda)


shape : torch.Size([2, 3])
dtype : torch.float32
device: cpu


**`shape`** tells you the size along each dimension. `torch.Size([2, 3])` means "2 rows, 3 columns" — or more generally, "dimension 0 has size 2, dimension 1 has size 3." You will check `.shape` constantly; it's the #1 debugging tool in PyTorch.

**`dtype`** tells you the numeric type. The two you'll see most:

| dtype | Meaning | Typical use |
|---|---|---|
| `torch.float32` | 32-bit decimal number | Default for model weights and inputs |
| `torch.int64` (`torch.long`) | 64-bit whole number | Class labels for classification |

**Rule to remember now, understand later:** mixing `float32` model weights with `int64` labels in the wrong place is a common source of errors. `nn.CrossEntropyLoss`, for instance, expects predictions as `float` and labels as `long` — we'll hit this directly in a later module.

**`device`** tells you whether the tensor lives in ordinary system memory (`cpu`) or GPU memory (`cuda`). Operations between two tensors require them to be on the *same* device — this is the single most common beginner error, and you'll deliberately trigger it in the exercise below so you recognize it instantly in the future.


### 🔮 Predict before you run

Before running the next cell, predict: what will `x.shape` be for `torch.tensor([[1, 2, 3, 4]])` — note the *extra* pair of brackets around the whole thing?


In [4]:
x = torch.tensor([[1, 2, 3, 4]])
print(x.shape)


torch.Size([1, 4])


If you predicted `torch.Size([1, 4])` — one row, four columns — you're already reading shapes correctly. The outer brackets create dimension 0 (size 1), and the inner list creates dimension 1 (size 4). This distinction between `[1, 2, 3, 4]` (shape `(4,)`) and `[[1, 2, 3, 4]]` (shape `(1, 4)`) trips up nearly everyone at first, and matters a lot once you start feeding batches of data into models.


## 🐛 Debugging Challenge: The Device Mismatch

This is, by a wide margin, the error you will see most often as a beginner. Let's produce it on purpose so you recognize it instantly later.


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

a = torch.randn(3, 3)              # created on CPU by default
b = torch.randn(3, 3).to(device)   # explicitly moved to `device`

try:
    result = a + b
except RuntimeError as e:
    print("RuntimeError caught:")
    print(e)


Using device: cuda


c:\Users\USERAS\ANACONDA\envs\ditto_train\lib\site-packages\torch\cuda\__init__.py:218: UserWarning: 
NVIDIA GeForce RTX 5060 Ti with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5060 Ti GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


RuntimeError caught:
Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!


If you're running this on a machine *without* a GPU, `device` is `"cpu"` for both tensors, so no error occurs — that's expected and fine; the error only appears on a GPU-enabled machine. Either way, the fix is the same: **move every tensor involved in an operation onto the same device before computing**, typically by calling `.to(device)` right after creating or loading each tensor.

```python
a = a.to(device)
b = b.to(device)
result = a + b   # now safe
```


## Reshaping: Same Data, Different Shape

Reshaping doesn't move or copy the underlying numbers — it just changes how they're *grouped* into dimensions. Think of it as relabeling, not rearranging.


In [6]:
x = torch.arange(12)   # tensor([0, 1, 2, ..., 11]), shape (12,)
print("original:", x.shape)

y = x.reshape(3, 4)     # same 12 numbers, viewed as 3 rows of 4
print("reshaped:", y.shape)
print(y)

z = x.reshape(2, 2, 3)  # same 12 numbers, viewed as 2 blocks of 2x3
print("reshaped again:", z.shape)


original: torch.Size([12])
reshaped: torch.Size([3, 4])
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
reshaped again: torch.Size([2, 2, 3])


`reshape` requires the total number of elements to stay the same — 12 elements can become `(3, 4)`, `(2, 2, 3)`, or `(12,)`, but never `(3, 5)` (15 slots for 12 elements). Try it below and see the error PyTorch gives you.


In [7]:
try:
    bad = x.reshape(3, 5)
except RuntimeError as e:
    print(e)


shape '[3, 5]' is invalid for input of size 12


You'll also often see `x.unsqueeze(dim)` (insert a size-1 dimension) and `x.squeeze()` (remove size-1 dimensions) — these come up constantly when a function expects a "batch" dimension that a single example doesn't naturally have.


In [8]:
single_image = torch.randn(28, 28)          # one 28x28 grayscale image
print("before:", single_image.shape)

batched = single_image.unsqueeze(0)          # add a batch dimension at position 0
print("after unsqueeze(0):", batched.shape)  # now looks like "a batch of 1 image"

back = batched.squeeze(0)                    # remove it again
print("after squeeze(0):", back.shape)


before: torch.Size([28, 28])
after unsqueeze(0): torch.Size([1, 28, 28])
after squeeze(0): torch.Size([28, 28])


This matters because PyTorch models almost always expect a **batch dimension**, even if you're only passing in one example. `unsqueeze(0)` is how you turn "one image" into "a batch containing one image."


## Exercises

🟢 **Beginner:** Create a 1D tensor of the numbers 0 through 9 using `torch.arange(10)`. Print its shape, dtype, and device.

🟡 **Intermediate:** Reshape that tensor into shape `(2, 5)`, then into `(5, 2)`. Before running each reshape, predict what the printed tensor will look like.

🔴 **Challenge:** Create a tensor of shape `(3, 4)` using `torch.randn`. Without using `.shape`, write code that computes the total number of elements in the tensor using only `.size()` (hint: `.size()` returns the same info as `.shape`, but as a callable — look at what `torch.Size` supports, e.g. indexing and multiplication).


In [9]:
# Space for your exercise solutions



## Common Mistakes

- **Confusing `torch.Tensor([2,3])` (values 2 and 3) with `torch.zeros(2,3)` (a 2x3 grid of zeros).** Passing data vs. passing a shape are very different calls — watch the function name.
- **Forgetting a tensor needs a batch dimension** before passing it to a model — leads to a shape-mismatch error deep inside the model rather than an obvious one.
- **Assuming reshape can create or destroy elements.** It can't — the total element count must match before and after.
- **Comparing/combining tensors on different devices.** Always `.to(device)` before combining.

## Mental Model

A tensor is "a box of numbers plus a label describing the box's shape, its number type, and which machine (CPU/GPU) it's sitting on." Whenever something breaks, check the label first — `.shape`, `.dtype`, `.device` — before suspecting the math.

## Key Takeaways

- PyTorch exists to combine array computation + automatic differentiation + GPU support in one object: the tensor.
- Every tensor has a shape, dtype, and device — check all three when debugging.
- Reshaping relabels data; it never creates, destroys, or moves individual values.
- Operations require matching devices; this is the single most common beginner error.

## What's Next

**Module 02 — Tensor Operations, Indexing & Broadcasting** builds directly on this: you'll learn how tensors of *different* shapes can still be combined via broadcasting, and how to slice into specific parts of a tensor.

## Checklist

- [ ] I can create tensors from Python lists and using `torch.zeros`/`torch.ones`/`torch.randn`
- [ ] I can explain what shape, dtype, and device each mean
- [ ] I can reshape a tensor and predict whether a given reshape is valid
- [ ] I understand why operations require tensors to share the same device
